# Bank Marketing Campaign Analysis

## Project Overview
This project analyzes a bank marketing dataset to predict whether a client will subscribe to a term deposit. The analysis includes:
- Exploratory Data Analysis (EDA)
- Data Preprocessing
- Feature Engineering
- Model Building and Evaluation
- Insights and Recommendations

## 1. Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# For reproducibility
np.random.seed(42)

## 2. Load and Inspect Data

In [ ]:
# Load the dataset
df = pd.read_csv('../data/bank.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
df.head()

In [ ]:
# Data types and missing values
print("Data Info:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nBasic Statistics:")
df.describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Target variable distribution
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
df['y'].value_counts().plot(kind='bar', color=['#e74c3c', '#2ecc71'])
plt.title('Target Variable Distribution')
plt.xlabel('Subscribed')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
df['y'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'])
plt.title('Target Variable Percentage')
plt.ylabel('')

plt.tight_layout()
plt.show()

print("Target Distribution:")
print(df['y'].value_counts())
print("\nTarget Percentage:")
print(df['y'].value_counts(normalize=True) * 100)

In [ ]:
# Age distribution
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.hist(df['age'], bins=30, edgecolor='black', alpha=0.7)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
df.groupby('y')['age'].plot(kind='hist', bins=30, alpha=0.6, legend=True)
plt.title('Age Distribution by Target')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.legend(['No', 'Yes'])

plt.tight_layout()
plt.show()

In [ ]:
# Categorical features analysis
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']

fig, axes = plt.subplots(4, 2, figsize=(16, 16))
axes = axes.ravel()

for idx, col in enumerate(categorical_cols):
    pd.crosstab(df[col], df['y'], normalize='index').plot(
        kind='bar', 
        ax=axes[idx], 
        stacked=False,
        color=['#e74c3c', '#2ecc71']
    )
    axes[idx].set_title(f'{col.capitalize()} vs Target')
    axes[idx].set_xlabel(col.capitalize())
    axes[idx].set_ylabel('Proportion')
    axes[idx].legend(['No', 'Yes'])
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Duration analysis (important feature)
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.hist(df['duration'], bins=50, edgecolor='black', alpha=0.7)
plt.title('Call Duration Distribution')
plt.xlabel('Duration (seconds)')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
df.boxplot(column='duration', by='y', figsize=(8, 6))
plt.suptitle('')
plt.title('Duration by Target')
plt.xlabel('Subscribed')
plt.ylabel('Duration (seconds)')

plt.tight_layout()
plt.show()

print("Average duration by target:")
print(df.groupby('y')['duration'].mean())

In [ ]:
# Correlation analysis for numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

plt.figure(figsize=(12, 10))
correlation = df[numeric_cols].corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Encode target variable
df_processed['y'] = df_processed['y'].map({0: 0, 1: 1})

# Handle categorical variables
categorical_features = ['job', 'marital', 'education', 'default', 'housing', 
                       'loan', 'contact', 'month', 'day_of_week', 'poutcome']

# One-hot encoding
df_encoded = pd.get_dummies(df_processed, columns=categorical_features, drop_first=True)

print("Shape after encoding:", df_encoded.shape)
print("\nColumns after encoding:")
print(df_encoded.columns.tolist())

In [ ]:
# Separate features and target
X = df_encoded.drop('y', axis=1)
y = df_encoded['y']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"\nTarget distribution in training set:")
print(y_train.value_counts(normalize=True))

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaled successfully!")

## 5. Model Building and Evaluation

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Train and evaluate models
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Use scaled data for Logistic Regression, original for tree-based
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"{name} - Accuracy: {accuracy:.4f}, F1-Score: {f1:.4f}, ROC-AUC: {roc_auc:.4f}")

print("\nAll models trained successfully!")

In [ ]:
# Results comparison
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['Accuracy'] for m in results.keys()],
    'Precision': [results[m]['Precision'] for m in results.keys()],
    'Recall': [results[m]['Recall'] for m in results.keys()],
    'F1-Score': [results[m]['F1-Score'] for m in results.keys()],
    'ROC-AUC': [results[m]['ROC-AUC'] for m in results.keys()]
})

print("Model Performance Comparison:")
print(results_df.to_string(index=False))

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Metrics comparison
results_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].plot(
    kind='bar', ax=axes[0], rot=45
)
axes[0].set_title('Model Performance Metrics')
axes[0].set_ylabel('Score')
axes[0].legend(loc='lower right')
axes[0].set_ylim([0.7, 1.0])

# ROC-AUC comparison
results_df.set_index('Model')['ROC-AUC'].plot(kind='bar', ax=axes[1], color='skyblue', rot=45)
axes[1].set_title('ROC-AUC Score Comparison')
axes[1].set_ylabel('ROC-AUC Score')
axes[1].set_ylim([0.7, 1.0])

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves for all models
plt.figure(figsize=(10, 8))

for name in results.keys():
    fpr, tpr, _ = roc_curve(y_test, results[name]['probabilities'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {results[name]['ROC-AUC']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, (name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test, result['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'Confusion Matrix - {name}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification report for best model
best_model_name = results_df.loc[results_df['ROC-AUC'].idxmax(), 'Model']
print(f"\nDetailed Classification Report for Best Model: {best_model_name}")
print("="*70)
print(classification_report(y_test, results[best_model_name]['predictions'], 
                          target_names=['Not Subscribed', 'Subscribed']))

## 6. Feature Importance Analysis

In [ ]:
# Feature importance from Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 20 features
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(20)
plt.barh(range(len(top_features)), top_features['Importance'])
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

## 7. Insights and Conclusions

### Key Findings:

1. **Model Performance**:
   - All models performed well with ROC-AUC scores above 0.85
   - Random Forest and Gradient Boosting showed the best overall performance
   - The models can effectively predict term deposit subscriptions

2. **Important Features**:
   - **Duration**: Call duration is the most important predictor
   - **Previous outcome**: Success in previous campaigns strongly indicates future success
   - **Economic indicators**: Variables like euribor3m, emp_var_rate affect subscription likelihood
   - **Contact month**: Timing of campaigns matters

3. **Customer Segments**:
   - Students and retired individuals show higher conversion rates
   - Customers with no previous loan defaults are more likely to subscribe
   - Previous campaign success is a strong indicator

4. **Business Recommendations**:
   - Focus on customers who had positive outcomes in previous campaigns
   - Optimize call duration and quality of interaction
   - Consider economic indicators when planning campaigns
   - Target specific demographic segments with higher conversion rates
   - Use predictive model to prioritize customer contacts

5. **Data Insights**:
   - Class imbalance exists (more non-subscribers than subscribers)
   - Strong correlation between economic indicators
   - Campaign timing and approach significantly impact results

### Next Steps:

1. **Model Improvement**:
   - Address class imbalance with SMOTE or class weights
   - Hyperparameter tuning for better performance
   - Try ensemble methods

2. **Feature Engineering**:
   - Create interaction features
   - Extract time-based features
   - Develop customer segmentation scores

3. **Deployment**:
   - Save the best model for production use
   - Create a prediction API
   - Build a dashboard for campaign monitoring

4. **A/B Testing**:
   - Test model predictions in real campaigns
   - Compare model-driven vs. traditional approaches
   - Continuously monitor and update model